# Chapter 9 &mdash; Closure Results for Regular Languages

**Concept 8 of the Chapter 9 decomposition:** *Closure Results for Regular Languages*

Closed under union, concatenation, star, complement, intersection, reversal and homomorphism &mdash; each by a construction.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Closure-Results/Concept-Closure-Results.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Regular languages are **closed** under every operation in this book so far, and each
closure is proved by exhibiting a construction:

| operation | construction | chapter |
|---|---|---|
| union, concatenation, star | RE operators / Thompson | 8 |
| union, intersection | product construction | 6 |
| complement | totalize, then flip $F$ | 6 |
| reversal | flip the arrows &rarr; NFA | 7 |
| homomorphism | rename symbols (`shomo`) | 9 |

Closure is what lets you **decompose a design**: build the pieces, combine them, and
know the result is still regular &mdash; and still recognisable by a machine you can
actually build.

## 2. Definitions

### The constructions, as one-liners

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))

A = re_dfa("(0+1)*01")        # ends in 01
B = re_dfa("(0+1)*11")        # ends in 11

### Reference predicates

In [ ]:
from itertools import product
STRS = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
inA  = lambda s: s.endswith('01')
inB  = lambda s: s.endswith('11')
def check(D, f, name):
    bad = [s for s in STRS if accepts_dfa(D, s) != f(s)]
    print("%-26s mismatches: %d" % (name, len(bad)))
    assert not bad

## 3. Tests

**Union** and **intersection**, by the product construction.

In [ ]:
check(min_dfa(union_dfa(A, B)),     lambda s: inA(s) or inB(s),  "union")
check(min_dfa(intersect_dfa(A, B)), lambda s: inA(s) and inB(s), "intersection")

**Complement**, by totalize-then-flip.

In [ ]:
check(min_dfa(comp_dfa(A)), lambda s: not inA(s), "complement")

**Concatenation** and **star**, via the RE operators.

In [ ]:
cat  = re_dfa("((0+1)*01)((0+1)*11)")
star = re_dfa("((0+1)*01)*")
print("concatenation minimal |Q| :", len(cat["Q"]))
print("star          minimal |Q| :", len(star["Q"]))
assert accepts_dfa(cat, '0111') and accepts_dfa(star, '')
assert accepts_dfa(star, '0101')

**Reversal**, by flipping the arrows.

In [ ]:
R = min_dfa(nfa2dfa(rev_dfa(A)))
check(R, lambda s: inA(s[::-1]), "reversal")
print("L(A) ends in 01, so L(A)^R starts with 10 :",
      accepts_dfa(R, '10'), accepts_dfa(R, '01'))

**Homomorphism** on strings, with `shomo`, and on machines with `apply_h_dfa`.

In [ ]:
h = lambda c: {'0': 'a', '1': 'b'}[c]
print("shomo('0101', h) =", shomo('0101', h))
assert shomo('0101', h) == 'abab'
H = apply_h_dfa(A, h)
Hd = min_dfa(nfa2dfa(H)) if 'Q0' in H else min_dfa(H)
print("image alphabet :", sorted(Hd["Sigma"]))
assert Hd["Sigma"] == {'a', 'b'}
assert accepts_dfa(Hd, 'ab') and not accepts_dfa(Hd, 'ba')
print("the renamed machine accepts exactly the renamed strings")

Closure is what makes **decomposition** safe.

In [ ]:
big = min_dfa(intersect_dfa(comp_dfa(A), union_dfa(B, re_dfa("0(0+1)*"))))
spec = lambda s: (not inA(s)) and (inB(s) or s.startswith('0'))
check(big, spec, "a three-operation combination")
print("\nBuilt from complement, union and intersection -- still regular, still a DFA.")

## 4. Animation

A combination of four closure operations, minimized.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(intersect_dfa(comp_dfa(A), B)), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Are regular languages closed under **subset**? Under **infinite union**?
2. Prove closure under set difference using only complement and intersection.
3. Which closure property did Chapter 4 use to settle $L_{if}$?

In [ ]:
# Your work for the exercises above.